# 01 - Pré-processamento

Pré-processamento do ficheiro *accepted* (Fase 2 do enunciado, *Pre-processing*).

**Ordem dos passos**

O enunciado pede: *"Define partitions before data-dependent preprocessing"*. Por isso, antes de olhar para valores em falta, outliers ou qualidade dos dados, faz-se só o que não depende de nenhuma estatística dos dados:
1. Remover as linhas que não são empréstimos (regra estrutural).
2. Separar as partições temporais definidas na Fase 1 (`docs/01_formulacao_problema/desenho_split.md`).

A partir daqui, todas as análises e decisões de limpeza usam **só o conjunto de desenvolvimento**. O teste final fica guardado à parte e só recebe as mesmas regras, sem as rever.

**Antes de correr:** é preciso ter corrido `00_conversao_dados.ipynb`. O ficheiro *accepted* ocupa cerca de 4 GB em memória.

In [ ]:
# Setup: imports e caminhos do projeto
import gc
import sys
from pathlib import Path

# Importar o pacote do projeto.
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import pandas as pd

from lending import config
from lending.cleaning import find_non_loan_rows
from lending.exclusions import ExclusionLog
from lending.parsing import parse_month_year
from lending.partition import TemporalPartitioner
from lending.quality import (DISGUISED_NEVER_PAIRS, DuplicateReport, MissingnessReport,
                             recode_disguised_missing)
from lending.schema import SchemaParser, SchemaReport
from lending.storage import PickleStore

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 1. Linhas que não são empréstimos

O CSV do Kaggle é a junção de várias exportações da LendingClub. Cada exportação termina com duas linhas de totais (`Total amount funded in policy code 1/2: ...`) e há também um separador (`Loans that do not meet the credit policy`). Estas linhas estão **a meio do ficheiro**, não só no fim.

**Regra:** o `id` não é um número inteiro **e** todas as outras colunas estão vazias. Se aparecer um `id` não numérico numa linha com dados, o código dá erro em vez de a remover, porque já não seria um rodapé.

A regra é estrutural, por isso aplica-se ao ficheiro inteiro, antes da partição. Cada remoção fica no registo de exclusões, que no fim dá a tabela de exclusões do relatório.

In [ ]:
accepted = PickleStore.load(config.ACCEPTED_RAW_PKL, expected_type=pd.DataFrame)
print(f"accepted: {len(accepted):,} linhas x {accepted.shape[1]} colunas")

non_loan = find_non_loan_rows(accepted)
print(f"Linhas que não são empréstimos: {non_loan.sum()}")

# O texto de cada linha marcada e quantas vezes aparece
accepted.loc[non_loan, "id"].str.replace(r"\d+$", "<valor>", regex=True).value_counts()

In [ ]:
# Posição das linhas no ficheiro: confirma que estão espalhadas, entre exportações
accepted.loc[non_loan, ["id"]]

In [ ]:
log = ExclusionLog("accepted -> desenvolvimento")

# Só se regista: filtrar o DataFrame inteiro criaria uma cópia de ~4 GB.
# As linhas que ficam são escolhidas na partição (passo 2).
log.record(
    step="Linhas que não são empréstimos",
    kind="remoção",
    rule="id não numérico e todas as outras colunas vazias",
    reason="Totais e separadores das exportações da LendingClub, não são empréstimos",
    rows_before=len(accepted),
    rows_removed=int(non_loan.sum()),
)
log.to_frame()

## 2. Partição temporal

**Data usada:** `issue_d`, o mês de financiamento (texto `Mon-AAAA`, ex.: `Dec-2015`). A conversão para data é **determinística**: uma tabela fixa de meses, sem estimar nada a partir dos dados. Por isso pode ser feita antes da partição (é aliás necessária para a fazer). Um valor fora do formato dá erro.

**Cortes** (`config.PARTITIONS`, decididos na Fase 1):

| Partição | Período (`issue_d`) | Utilização |
|---|---|---|
| Desenvolvimento | 2007-06 a 2014-12 | Treino, validação, EDA, testes de hipóteses, *fit* das transformações |
| Teste final | 2015-01 a 2015-12 | Avaliação final, uma única vez, no fim da Parte 2 |
| Reservado | 2016-01 a 2018-12 | Não usado na Parte 1 |

Os cortes são feitos ao mês, por isso os empréstimos do mesmo mês ficam sempre juntos. Cada linha tem de cair numa e uma só partição: uma data em falta ou fora dos períodos dá erro.

In [ ]:
# Datas só das linhas que são empréstimos
issue_date = parse_month_year(accepted.loc[~non_loan, "issue_d"])
print(f"issue_d: {issue_date.min():%Y-%m} a {issue_date.max():%Y-%m}, "
      f"{issue_date.isna().sum()} em falta")

partitioner = TemporalPartitioner()
partitioner.summary(issue_date)

In [ ]:
# Empréstimos por ano de financiamento e partição
# (só contagens: não se olha para nenhum valor do teste nem do reservado)
pd.crosstab(issue_date.dt.year.rename("ano"), partitioner.assign(issue_date).rename("partição"),
            margins=True, margins_name="Total")

In [ ]:
# Só se copiam para memória as partições usadas; o reservado continua no pickle original
parts = partitioner.split(accepted, issue_date, names=["development", "test"])
accepted_dev, accepted_test = parts["development"], parts["test"]

sizes = partitioner.summary(issue_date)["rows"]
assert len(accepted_dev) == sizes["development"] and len(accepted_test) == sizes["test"]

# Cada empréstimo aparece uma só vez no ficheiro: nenhum pode ficar em duas partições
assert accepted.loc[~non_loan, "id"].is_unique
print(f"ids únicos no ficheiro inteiro: {(~non_loan).sum():,} empréstimos")
# O identificador do mutuário não existe na exportação pública (usado na secção 7)
print(f"member_id preenchido no ficheiro inteiro: {accepted['member_id'].notna().sum()}")

log.record(
    step="Separar o teste final",
    kind="partição",
    rule="issue_d de 2015-01 a 2015-12",
    reason="Teste final cronológico, guardado à parte para a avaliação final (desenho_split.md)",
    rows_before=log.rows_after,
    rows_removed=int(sizes["test"]),
)
log.record(
    step="Separar o período reservado",
    kind="partição",
    rule="issue_d de 2016-01 a 2018-12",
    reason="Não usado na Parte 1: os empréstimos de 36 meses ainda não tinham terminado em 2018-12",
    rows_before=log.rows_after,
    rows_removed=int(sizes["reserved"]),
)
assert log.rows_after == len(accepted_dev)

# Libertar o ficheiro completo: a partir daqui só se usa o desenvolvimento
del accepted, parts, non_loan, issue_date
_ = gc.collect()

log.to_frame()

## 3. Guardar as partições

As partições são guardadas **ainda em texto**, sem nenhum valor alterado: este passo só separa linhas. A conversão de tipos e a limpeza vêm a seguir, sobre o desenvolvimento.

- `accepted_dev_raw.pkl`: conjunto de desenvolvimento.
- `accepted_test_raw.pkl`: teste final. **Não volta a ser aberto** até à avaliação final; nessa altura recebe as mesmas regras de limpeza, sem as rever.
- `accepted_dev_exclusions.pkl`: o registo de exclusões, que os passos seguintes continuam.

O período reservado não é guardado: não é usado na Parte 1 e continua disponível em `accepted_raw.pkl`.

In [ ]:
PickleStore.save(accepted_dev, config.ACCEPTED_DEV_RAW_PKL)
PickleStore.save(accepted_test, config.ACCEPTED_TEST_RAW_PKL)
PickleStore.save(log, config.ACCEPTED_DEV_LOG_PKL)

print(f"Desenvolvimento: {len(accepted_dev):,} linhas -> {config.ACCEPTED_DEV_RAW_PKL.name}")
print(f"Teste final:     {len(accepted_test):,} linhas -> {config.ACCEPTED_TEST_RAW_PKL.name}")
print(f"Registo:         {log} -> {config.ACCEPTED_DEV_LOG_PKL.name}")

# O teste sai da memória: a partir daqui não é usado
del accepted_test
_ = gc.collect()

## 4. Conversão de tipos (parsing determinístico)

Até aqui todas as colunas são texto. Cada coluna é agora convertida para o seu tipo, segundo o schema declarado em `src/lending/schema.py`, onde o tipo de cada uma das 151 colunas está escrito à mão em vez de ser adivinhado pelo pandas.

| Tipo | Exemplo no ficheiro | Resultado | Colunas |
|---|---|---|---:|
| `id` | `"9006963"` | inteiro | 1 |
| `number` | `"10000.0"`, `"12.99"`, `"1e-10"` | decimal (`float64`) | 112 |
| `term` | `" 36 months"` | `36.0` (meses) | 1 |
| `emp_length` | `"< 1 year"`, `"10+ years"` | `0` a `10` (anos) | 1 |
| `month_year` | `"Dec-2015"` | `2015-12-01` | 11 |
| `flag` | `"Y"`/`"N"`, `"y"`/`"n"` | `True`/`False` | 3 |
| `category` | `"MORTGAGE"`, `"B3"` | texto sem espaços nas pontas | 17 |
| `text` | `"Teacher"`, url, `desc` | texto sem espaços nas pontas | 5 |

**Porque pode ser feito antes de qualquer *fit*:** cada valor é convertido só a partir de si próprio, com uma regra fixa. Nada é estimado a partir dos dados, por isso o mesmo `SchemaParser` vai converter o teste final exatamente da mesma forma, sem o rever. É o caso que o enunciado dá como exemplo: *"removing a percent sign or converting a date string using a fixed format"*.

**Garantias:**
- Um valor com formato inesperado dá **erro**, nunca passa a NaN em silêncio.
- Nenhum valor preenchido se perde. A única exceção é o texto só com espaços, que é um campo vazio e passa a valor em falta (fica contado na tabela abaixo).
- Não se mudam maiúsculas nem se juntam categorias (ex.: `"RN"` e `"Registered Nurse"`): isso é uma decisão de limpeza, não de parsing.

In [ ]:
parser = SchemaParser()
accepted_dev_parsed = parser.parse(accepted_dev)
parse_summary = parser.summary(accepted_dev, accepted_dev_parsed)

print(f"{accepted_dev_parsed.shape[1]} colunas convertidas | memória: "
      f"{accepted_dev.memory_usage(deep=True).sum() / 1e6:,.0f} MB -> "
      f"{accepted_dev_parsed.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

# Quantas colunas de cada tipo e com que dtype ficaram
parse_summary.groupby(["kind", "dtype"]).size().rename("colunas").to_frame()

In [ ]:
# Colunas em que algum valor passou a valor em falta (só pode ser texto só com espaços)
parse_summary[parse_summary["blank_to_missing"] > 0]

In [ ]:
# Antes e depois, nas mesmas linhas (amostra aleatória com semente, só para ver a conversão)
example_columns = ["id", "term", "int_rate", "emp_length", "issue_d", "earliest_cr_line",
                   "revol_util", "pymnt_plan", "home_ownership"]
example_rows = accepted_dev.sample(5, random_state=config.RANDOM_SEED).index

pd.concat({"antes (texto)": accepted_dev.loc[example_rows, example_columns],
           "depois": accepted_dev_parsed.loc[example_rows, example_columns]}, axis=1)

In [ ]:
# As tabelas de conversão de term e emp_length: cada texto corresponde a um só número
# (dropna=False mostra também os valores em falta, que continuam em falta)
for col in ["term", "emp_length"]:
    mapping = (pd.DataFrame({"texto": accepted_dev[col], "valor": accepted_dev_parsed[col]})
               .value_counts(dropna=False).rename("linhas").reset_index()
               .sort_values("valor", na_position="last"))
    display(mapping.set_index("texto"))

In [ ]:
# A partir daqui trabalha-se com o desenvolvimento já convertido
# (a versão em texto continua guardada em accepted_dev_raw.pkl)
accepted_dev = accepted_dev_parsed
del accepted_dev_parsed
_ = gc.collect()

accepted_dev.dtypes.value_counts().rename("colunas").to_frame()

## 5. Schema e unidades

O enunciado pede para descrever *"the source files, schema, units"*. O **significado** de cada coluna já está na tabela de elegibilidade da Fase 1 (`docs/01_formulacao_problema/elegibilidade_colunas.md`). Aqui descreve-se, para cada coluna:
- o **tipo** depois da conversão e o `dtype`;
- a **unidade** (`ACCEPTED_UNITS` em `src/lending/schema.py`): USD, %, meses, pontos FICO, contagem, etc.;
- o **intervalo observado** no desenvolvimento, que serve para confirmar a unidade. Exemplo: `int_rate` entre 5,42 e 26,06 confirma que está em percentagem (12,99 = 12,99%) e não em fração de 0 a 1.

**Convenção das percentagens:** todas as colunas em % estão na escala 0–100, não 0–1. Algumas passam de 100 (ex.: `revol_util`), o que é analisado na qualidade dos dados.

A tabela completa é escrita em `docs/02_preprocessamento/schema_accepted.md`.

In [ ]:
schema_report = SchemaReport()
schema_description = schema_report.describe(accepted_dev)

# Quantas colunas há de cada unidade
schema_description["unit"].value_counts().rename("colunas").to_frame()

In [ ]:
# Confirmação das unidades: colunas em % e pontos FICO, com o intervalo observado
schema_description[schema_description["unit"].isin(["%", "pontos FICO"])
                   & (schema_description["distinct"] > 0)]

In [ ]:
# Datas: a última data registada em cada coluna mostra até quando o ficheiro foi atualizado
date_columns = schema_description.index[schema_description["kind"] == "month_year"]
last_dates = accepted_dev[date_columns].max().dt.strftime("%Y-%m").rename("última data")
last_dates.to_frame().assign(depois_de_2018=accepted_dev[date_columns].gt("2018-12-31").sum())

In [ ]:
# Tabela completa do schema (151 colunas) e registo em docs/
out = schema_report.write_markdown(schema_description,
                                   config.DOCS_DIR / "02_preprocessamento" / "schema_accepted.md",
                                   dataset="accepted, desenvolvimento", n_rows=len(accepted_dev))
print("Escrito:", out.relative_to(config.PROJECT_ROOT))

with pd.option_context("display.max_rows", 200):
    display(schema_description)

## 6. Valores em falta

O enunciado pede para descrever os *"missing values"* e para *"distinguish non-applicability from missing information"*. Por isso não chega contar: é preciso saber **porque** falta cada valor, porque cada razão pede um tratamento diferente.

| Significado | Exemplo | O que quer dizer | Tratamento adequado |
|---|---|---|---|
| **sem evento** | `mths_since_last_delinq` | Nunca houve um atraso | Indicar "sem evento"; imputar a mediana inventava um atraso |
| **sem denominador** | `revol_util` | Não há limite rotativo; o rácio não existe | Indicar "não aplicável" |
| **só alguns empréstimos** | `hardship_*`, `settlement_*` | Só existe para quem teve um plano ou acordo | Pós-concessão: excluídas das tarefas |
| **só pedidos conjuntos** | `dti_joint`, `sec_app_*` | Só existe em pedidos conjuntos | Não pôr a zero (enunciado) |
| **não registado** | `emp_length` | O valor existia mas não foi guardado | Imputação aprendida no treino, com indicador |

O significado está declarado em `MISSING_MEANING` (`src/lending/quality.py`), a partir do dicionário de dados e de verificações com outras colunas. Além disso, há colunas que a LendingClub **só começou a registar mais tarde**: estão vazias nos primeiros anos por uma razão estrutural.

Esta secção só **analisa e documenta**. A imputação e os indicadores são transformações aprendidas, por isso ficam no pipeline (com *fit* só no treino). A única alteração feita aqui é a correção determinística dos códigos disfarçados (6.1).

### 6.1 Códigos disfarçados: "nunca aconteceu" guardado como 0

Em 2007–2008, `mths_since_last_record` e `mths_since_last_delinq` quase não têm valores em falta, mas têm muitos zeros. Esses zeros aparecem só em mutuários com `pub_rec` = 0 (ou `delinq_2yrs` = 0). "0 meses desde o último registo público" é impossível para quem tem 0 registos públicos, e um atraso há 0 meses contaria nos últimos 2 anos (`delinq_2yrs`). A partir de 2009 o padrão desaparece: a LendingClub mudou a forma de codificar "nunca aconteceu" de 0 para vazio.

**Regra** (determinística, dentro de cada linha): meses = 0 **e** contagem = 0 → meses passa a valor em falta ("sem evento"). É o mesmo tipo de problema que o enunciado aponta no `Risk_Score` = 0 dos rejeitados.

In [ ]:
# Por ano: % em falta e % de zeros, e quantos zeros têm a contagem também a 0
issue_year = accepted_dev["issue_d"].dt.year.rename("ano")
disguised_check = {}
for months_col, count_col in DISGUISED_NEVER_PAIRS:
    zero = accepted_dev[months_col] == 0
    disguised_check[months_col] = pd.DataFrame({
        "% em falta": accepted_dev[months_col].isna().groupby(issue_year).mean() * 100,
        "% zeros": zero.groupby(issue_year).mean() * 100,
        f"zeros com {count_col}=0": (zero & (accepted_dev[count_col] == 0)).groupby(issue_year).sum(),
        f"zeros com {count_col}>0": (zero & (accepted_dev[count_col] > 0)).groupby(issue_year).sum(),
    }).round(1)
pd.concat(disguised_check, axis=1)

In [ ]:
accepted_dev, disguised_counts = recode_disguised_missing(accepted_dev)
disguised_counts.to_frame()

### 6.2 Resumo por coluna

- **status**: `completa` (sem valores em falta), `vazia` (100% em falta no desenvolvimento) ou `parcial`.
- **registered_from**: primeiro ano com pelo menos 1% dos valores preenchidos, ou seja, desde quando a coluna é registada. Um limiar mais alto confundiria "a coluna ainda não existia" com "o evento é raro".
- **meaning**: o significado do valor em falta (tabela acima).

In [ ]:
missingness = MissingnessReport()
missing_summary = missingness.summary(accepted_dev, accepted_dev["issue_d"])

# Quantas colunas de cada estado e significado
pd.crosstab(missing_summary["meaning"].replace("", "(sem valores em falta)"),
            missing_summary["status"], margins=True, margins_name="Total")

In [ ]:
# Colunas parciais, da que tem menos valores em falta para a que tem mais
with pd.option_context("display.max_rows", 200):
    display(missing_summary[missing_summary["status"] == "parcial"].sort_values("pct_missing"))

### 6.3 Colunas registadas mais tarde

Muitas colunas do *bureau* só começam a ser registadas em 2012. Antes disso estão vazias para **todos** os empréstimos, por uma razão estrutural (a LendingClub ainda não as guardava) e não por causa do mutuário. A tabela mostra a % em falta por ano de concessão.

In [ ]:
late_columns = missing_summary.index[(missing_summary["status"] == "parcial")
                                     & (missing_summary["registered_from"] > 2007)
                                     & (missing_summary["meaning"] != "só alguns empréstimos")]
print(f"{len(late_columns)} colunas só registadas depois de 2007, a partir de: "
      f"{missing_summary.loc[late_columns, 'registered_from'].value_counts().to_dict()}")

missing_by_year = missingness.by_year(accepted_dev, accepted_dev["issue_d"])
with pd.option_context("display.max_rows", 100):
    display(missing_by_year.loc[late_columns].round(1)
            .assign(registered_from=missing_summary.loc[late_columns, "registered_from"]))

### 6.4 Colunas com valores em falta exatamente nas mesmas linhas

Quando várias colunas faltam sempre juntas, vêm da mesma fonte (o mesmo relatório do *bureau*, o mesmo plano de *hardship*). Isto confirma as explicações acima e mostra que o tratamento deve ser feito por grupo.

In [ ]:
pattern_groups = missingness.identical_patterns(accepted_dev)
pd.DataFrame({
    "colunas": [len(g) for g in pattern_groups],
    "linhas em falta": [int(accepted_dev[g[0]].isna().sum()) for g in pattern_groups],
    "grupo": [", ".join(g) for g in pattern_groups],
})

### 6.5 Consequência para as tarefas

O treino da classificação é 2007-06 a 2011-12 (desenho da Fase 1). As colunas registadas só a partir de 2012 estão **100% vazias nesse treino**, por isso o classificador não pode aprender nada com elas. Entre elas estão vários *inputs* que o enunciado sugere (ex.: `tot_cur_bal`, `bc_util`, `mort_acc`). Na regressão (treino 2007–2014) existem a partir de 2012, mas faltam nos anos anteriores.

A tabela compara a % em falta dos *inputs* sugeridos pelo enunciado no treino da classificação e nos anos mais recentes do desenvolvimento.

In [ ]:
# Inputs base sugeridos pelo enunciado (secção "Note on data leakage")
suggested_inputs = [
    "emp_title", "emp_length", "home_ownership", "annual_inc", "addr_state", "zip_code", "dti",
    "fico_range_low", "fico_range_high", "earliest_cr_line", "delinq_2yrs", "inq_last_6mths",
    "open_acc", "total_acc", "pub_rec", "pub_rec_bankruptcies", "mort_acc", "revol_bal",
    "revol_util", "tot_cur_bal", "tot_hi_cred_lim", "bc_util", "bc_open_to_buy",
    "mo_sin_old_rev_tl_op", "num_actv_rev_tl", "pct_tl_nvr_dlq", "tax_liens",
]
class_train_window = accepted_dev["issue_d"] <= "2011-12-01"
recent_window = accepted_dev["issue_d"] >= "2013-01-01"

inputs_missing = pd.DataFrame({
    "% em falta 2007-2011 (treino classificação)":
        accepted_dev.loc[class_train_window, suggested_inputs].isna().mean() * 100,
    "% em falta 2013-2014": accepted_dev.loc[recent_window, suggested_inputs].isna().mean() * 100,
    "registered_from": missing_summary.loc[suggested_inputs, "registered_from"],
}).round(1).sort_values("% em falta 2007-2011 (treino classificação)", ascending=False)

unusable = inputs_missing.index[inputs_missing.iloc[:, 0] == 100]
print(f"{len(unusable)} dos {len(suggested_inputs)} inputs sugeridos estão 100% vazios no "
      f"treino da classificação: {list(unusable)}")
inputs_missing

## 7. Duplicados e mutuários repetidos

O enunciado pede: *"Address identifiable repeated borrowers and duplicate loans (but explain limitations when identifiers are unavailable)"*. São dois problemas diferentes:

- **Empréstimo duplicado:** o mesmo empréstimo em mais do que uma linha. Inflaciona os dados e, se as cópias caírem em partições diferentes, o modelo é avaliado com casos que já viu. Verifica-se pelo `id`, pelo `url` (que contém o `id`) e pelas restantes colunas.
- **Mutuário repetido:** a mesma pessoa com vários empréstimos. Não é um erro (são empréstimos reais), mas as linhas deixam de ser independentes. O identificador do mutuário (`member_id`) está vazio em todo o ficheiro, por isso só é possível uma **estimativa** com uma chave aproximada.

Na secção 2 confirmou-se que os `id` são todos diferentes no ficheiro inteiro, por isso nenhum empréstimo pode estar no desenvolvimento e no teste ao mesmo tempo.

In [ ]:
duplicates = DuplicateReport()

# Verificações de duplicados no desenvolvimento (rows = linhas que falham a verificação)
duplicates.checks(accepted_dev)

### 7.1 Estimativa de mutuários repetidos

Sem `member_id`, usa-se uma **chave aproximada** com atributos que mudam pouco entre dois pedidos da mesma pessoa (`REPEAT_BORROWER_KEY`): mês da primeira linha de crédito (`earliest_cr_line`, fixo para cada pessoa), localização (`zip_code`, `addr_state`) e perfil declarado (`annual_inc`, `emp_title`, `home_ownership`).

**Limitações da estimativa:**
- **Falsos negativos:** se a pessoa mudar de emprego, de rendimento ou de casa entre os dois pedidos, a chave muda e a repetição não é detetada.
- **Falsos positivos:** duas pessoas diferentes podem coincidir em todas as colunas (o `zip_code` só tem 3 dígitos).
- Linhas com alguma coluna da chave vazia (sobretudo `emp_title`) não entram.

Por isso o resultado é uma **ordem de grandeza**, não uma lista de mutuários.

In [ ]:
repeat_candidates = duplicates.repeat_borrowers(accepted_dev)
display(duplicates.repeat_summary(accepted_dev).to_frame("valor"))

# Empréstimos por grupo, e grupos com empréstimos no treino da classificação (até 2011)
# e depois dele: a mesma pessoa pode aparecer em lados diferentes de um corte temporal
group_dates = repeat_candidates.groupby("group")["issue_d"]
crosses_2012 = ((group_dates.min() <= "2011-12-01") & (group_dates.max() >= "2012-01-01")).sum()
print("empréstimos por grupo:", repeat_candidates.groupby("group").size().value_counts().sort_index().to_dict())
print(f"grupos com empréstimos até 2011 e a partir de 2012: {crosses_2012}")

# Exemplos (os primeiros grupos, ordenados pela chave)
repeat_candidates.head(8)